In [ ]:
from plant3dvision.colmap import ColmapRunner
from plant3dvision.tasks.colmap import CameraPoseQC
from plant3dvision.voxel_cuda import Backprojection
from plant3dvision.proc2d import linear
from plant3dvision.tasks.voxel_reconstruction import remap_averaging
from plant3dvision.tasks.voxel_reconstruction import origin_from_bounding_box
from plant3dvision.tasks.voxel_reconstruction import shape_from_bounding_box

from tqdm import tqdm
from imageio.v3 import imread
from plantdb.commons.test_database import test_database

In [ ]:
db = test_database('real_plant', no_auth=True)

In [ ]:
db.connect()

### Select the dataset to reconstruct

In [ ]:
dataset = db.get_scan("real_plant")

### Get the corresponding 'images' fileset

In [ ]:
images_fileset = dataset.get_fileset('images')
image_files = images_fileset.get_files()

## Binary masks

In [ ]:
masks = {}
for img_f in tqdm(image_files, unit='images'):
    img = imread(img_f.path())
    coeff_img = linear(img, [0.2, 1., 0.1], 'RGB')
    mask_img = coeff_img >= 0.2
    masks[img_f.id] = mask_img

## Colmap SfM reconstruction

In [ ]:
args = {"feature_extractor": {"--ImageReader.single_camera": "1"}}
colmap = ColmapRunner(image_files, matcher_method="exhaustive",
                      align_pcd=True, all_cli_args=args, colmap_exe="roboticsmicrofarms/colmap:3.8")

In [ ]:
points, images, cameras, sparse_pcd, dense_pcd, bounding_box = colmap.run()

## Pose estimation quality check

In [ ]:
cam_qc = CameraPoseQC(image_files, 3)

In [ ]:
outlier_dict = cam_qc.flag_outlier_poses()

In [ ]:
outlier_ids = [img for img, v in outlier_dict.items() if v]
print(f"Detected {len(outlier_ids)} potentially mis-estimated poses")

In [ ]:
cam_qc.plot_boxplot_estimation_distance()

In [ ]:
cam_qc.plot_pose_estimation_figure()

## Voxels reconstruction

In [ ]:
bounding_box = {"x": [300, 435], "y": [300, 435], "z": [-200, 100]}
voxel_size = 0.6

In [ ]:
shape = shape_from_bounding_box(bounding_box, voxel_size)
origin = origin_from_bounding_box(bounding_box)

In [ ]:
bp_averaging = Backprojection(shape, origin, voxel_size, type="averaging", labels=None, log=True)

In [ ]:
volume = bp_averaging.process_fileset(masks, "colmap_camera", False)